This part of the pipeline produces the UMAP plots as well as the accumulation curves of all pangenomes.

### Paths and parameters

#### Pipeline input folders

In [ ]:
pa.file = "./03-pangenomes/all/gene_presence_absence.Rtab"

group.ann.file = "./02-GTDB/filtered_classification_table"

plot_umap = "./utils/plot_umap.R"

#### Pipeline output folders

In [ ]:
task_root = "./05-pangenome-postprocessing"
system(paste0('mkdir -p ', task_root), intern = TRUE)

#### Tool pointers and parameters

Importing the `plot_umap` function as it's not officially released yet.

In [ ]:
source(plot_umap)
environment(plot_umap) = asNamespace('panstripe')

In [ ]:
set.seed(127)

In [ ]:
library(panstripe)
library(ape)
library(ggplot2)
library(umap)

### Load files and metadata

#### Presence/absence files

In [ ]:
pa = read_rtab(pa.file)

In [ ]:
nrow(pa)

#### Group annotations

In [ ]:
group.ann = read.table(group.ann.file, sep="\t", col.names = c('accession', 'group'), comment.char="")
group.ann

In [ ]:
group.ann.fac = as.factor(group.ann[match(rownames(pa), group.ann$accession),]$group)

### Plotting pangenome curves

#### UMAP plots

In [ ]:
svg(paste(task_root, 'panstripe_umap.svg', sep = "/"))
plot_umap(pa, category = group.ann.fac)
dev.off()

### Partitioning pangenomes

In [ ]:
core_accessory_threshold = 0.9
accessory_unique_threshold = 1/nrow(pa)

In [ ]:
partition_size = function(pa, ca, au){
    n.genomes = nrow(pa)
   
    n.core = sum(colSums(pa) >= n.genomes*ca)
    n.acc = sum(colSums(pa) < n.genomes*ca & colSums(pa) > 1)
    n.unique = sum(colSums(pa) <= 1)
    
    return(c(n.core, n.acc, n.unique))
}

In [ ]:
partition_size(pa, core_accessory_threshold, accessory_unique_threshold)

In [4]:
sessionInfo()

R version 4.3.2 (2023-10-31)
Platform: x86_64-pc-linux-gnu (64-bit)
Running under: Ubuntu 22.04.4 LTS

Matrix products: default
BLAS:   /usr/lib/x86_64-linux-gnu/openblas-pthread/libblas.so.3 
LAPACK: /usr/lib/x86_64-linux-gnu/openblas-pthread/libopenblasp-r0.3.20.so;  LAPACK version 3.10.0

locale:
 [1] LC_CTYPE=en_US.UTF-8       LC_NUMERIC=C              
 [3] LC_TIME=en_US.UTF-8        LC_COLLATE=en_US.UTF-8    
 [5] LC_MONETARY=en_US.UTF-8    LC_MESSAGES=en_US.UTF-8   
 [7] LC_PAPER=en_US.UTF-8       LC_NAME=C                 
 [9] LC_ADDRESS=C               LC_TELEPHONE=C            
[11] LC_MEASUREMENT=en_US.UTF-8 LC_IDENTIFICATION=C       

time zone: Europe/Brussels
tzcode source: system (glibc)

attached base packages:
[1] stats     graphics  grDevices utils     datasets  methods   base     

other attached packages:
[1] umap_0.2.10.0   ggplot2_3.5.0   ape_5.7-1       panstripe_0.2.0

loaded via a namespace (and not attached):
 [1] Matrix_1.6-3      gtable_0.3.4      jsonlite_